<div align="center">

# Data Projects and Hackathon 3  
## Project 
Sergio Fernandez, Alessandro Mecchia 

</div>

In [ ]:
import pandas as pd
import json
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

from pathlib import Path
import subprocess
import sys
import time

try:
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
except ModuleNotFoundError:
    print(f"pyarrow non trovato nel kernel corrente: {sys.executable}")
    print("Installazione di pyarrow nel kernel corrente...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])
    import pyarrow as pa
    import pyarrow.json as paj
    import pyarrow.parquet as pq
    
import pyarrow.parquet as pq

import pyarrow.parquet as pq
import ipywidgets as widgets
from IPython.display import display


## Data Creation 

da eseguire una sola volta 

In [ ]:
DATA_DIR = Path("data")
SOURCE_PATH = DATA_DIR / "DBLP-Citation-network-V18.jsonl"
TARGET_PATH = DATA_DIR / "DBLP-Citation-network-V18.parquet"
BLOCK_SIZE = 64 * 1024 * 1024  # 64 MiB per batch

if not SOURCE_PATH.exists():
    raise FileNotFoundError(f"File non trovato: {SOURCE_PATH}")

if pa.Codec.is_available("zstd"):
    COMPRESSION = "zstd"
elif pa.Codec.is_available("snappy"):
    COMPRESSION = "snappy"
else:
    COMPRESSION = None

print(f"Input : {SOURCE_PATH} ({SOURCE_PATH.stat().st_size / 1024**3:.2f} GiB)")
print(f"Output: {TARGET_PATH}")
print(f"Compressione: {COMPRESSION}")
print(f"Block size: {BLOCK_SIZE / 1024**2:.0f} MiB")

In [ ]:
reader = paj.open_json(
    SOURCE_PATH,
    read_options=paj.ReadOptions(block_size=BLOCK_SIZE),
)

writer = None
rows_written = 0
batches_written = 0
started_at = time.perf_counter()

try:
    if TARGET_PATH.exists():
        TARGET_PATH.unlink()

    while True:
        try:
            batch = reader.read_next_batch()
        except StopIteration:
            break

        if writer is None:
            writer = pq.ParquetWriter(
                TARGET_PATH,
                batch.schema,
                compression=COMPRESSION,
            )

        writer.write_batch(batch)
        rows_written += batch.num_rows
        batches_written += 1

        if batches_written % 25 == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"Batch: {batches_written:>5} | Rows: {rows_written:>12,} | Elapsed: {elapsed:>8.1f}s"
            )

    if writer is None:
        raise RuntimeError("Il file JSONL sembra vuoto: nessun batch letto.")
finally:
    reader.close()
    if writer is not None:
        writer.close()

elapsed = time.perf_counter() - started_at
source_size_gib = SOURCE_PATH.stat().st_size / 1024**3
target_size_gib = TARGET_PATH.stat().st_size / 1024**3

print()
print(f"Conversione completata in {elapsed:.1f}s")
print(f"Righe scritte : {rows_written:,}")
print(f"JSONL size    : {source_size_gib:.2f} GiB")
print(f"Parquet size  : {target_size_gib:.2f} GiB")

## Data Exploration 

In [ ]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")

print(f"Total rows   : {pf.metadata.num_rows:,}")
print(f"Columns      : {pf.metadata.num_columns}")
print(f"Row groups   : {pf.metadata.num_row_groups}")

In [ ]:
for i, name in enumerate(pf.schema_arrow.names):
    print(f"{i}. {name}")

In [ ]:
pf = pq.ParquetFile(r"data\DBLP-Citation-network-V18.parquet")
columns = pf.schema_arrow.names

dropdown = widgets.Dropdown(options=columns, description="Colonna:")
output = widgets.Output()

def on_change(change):
    if change["type"] == "change" and change["name"] == "value":
        with output:
            output.clear_output()
            batch = next(pf.iter_batches(batch_size=10, columns=[change["new"]]))
            df = batch.to_pandas()
            display(df[change["new"]])

dropdown.observe(on_change)
display(dropdown, output)